In [1]:
from transformers import AutoTokenizer
import json
from datasets import Dataset
import numpy as np
# from skmultilearn.model_selection import iterative_train_test_split
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [2]:
# Load the trained BERTopic model folder
# must be in GPU environment
topic_model = BERTopic.load("PVBERTopic/my_bertopic_model_HDBSCAN")

Wed Oct 15 17:05:10 2025 Building and compiling search function


In [3]:
file_path1 = "data/stratified_test_data_pinfo.json"  # Update with your actual file path
with open(file_path1, "r") as f:
    data1 = json.load(f)
file_path2 = "data/stratified_train_data_pinfo.json"  # Update with your actual file path
with open(file_path2, "r") as f:
    data2 = json.load(f)

In [4]:
len(data1 + data2)

837

In [5]:
len(data1)

192

In [6]:
len(data2)

645

In [7]:
# just test
try_txt = [data1[2]["text"], data1[1]["text"]]
# Example new sentence
new_sentence = try_txt
print (new_sentence)
# Transform it with your trained topic model
new_topics, new_probs = topic_model.transform(new_sentence)

# Get the assigned topic ID
for i in range(len(new_topics)):
    assigned_topic = new_topics[i]
    print(f"Assigned topic: {assigned_topic}")
    
    # Get the top 3 words for this topic
    if assigned_topic != -1:
        topic_words = topic_model.get_topic(assigned_topic)
        top_3_words = [word for word, _ in topic_words[:3]]
        print(f"Top 3 words for topic {assigned_topic}: {top_3_words}")
    else:
        print("This sentence was classified as outlier topic (-1). No top words.")

['From Provider: Salon pas lidocine patch to the affected area, it may take a few days to work, but use as directed', 'From Patient: Hi Dr. Person1,I left two messages last week on MM/DD/YYYY and MM/DD/YYYY. Asking about stopping the eliquis for the second time on this month since I had the first procedure of diagnostic lumbar facet block.RF ABLATION.AND DR.Person 2 wants to know for my second dose is scheduled for MM/DD/YYYY he wants to know if I cud have it done that date or cud schedule it a bit further date,since I have to stop the eliquis three days prior to the procedure. Thank you, Person2']


Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
Assigned topic: 296
Top 3 words for topic 296: ['lidocaine', 'patches', 'lidoderm']
Assigned topic: 188
Top 3 words for topic 188: ['eliquis', 'plavix', 'aspirin']


In [8]:
# still a try
def get_topics(sentences):
    assert type(sentences) == list
    # Transform it with your trained topic model
    new_topics, new_probs = topic_model.transform(sentences)
    # Get the assigned topic ID
    outlier_count = 0
    ret = []
    for i in range(len(new_topics)):
        assigned_topic = new_topics[i]
        #print(f"Assigned topic: {assigned_topic}")
        # Get the top 3 words for this topic
        if assigned_topic != -1:
            topic_words = topic_model.get_topic(assigned_topic)
            top_3_words = [word for word, _ in topic_words[:3]]
            ret.append(top_3_words)
            #print(f"Top 3 words for topic {assigned_topic}: {top_3_words}")
        else:
            outlier_count += 1
            ret.append([])
    #print (f"number of outliers is {outlier_count}")
    return ret
sents = [d["text"] for d in data2]
rets = get_topics(sents)
rets[1:3]

Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs


[['pharmacy', 'prescription', 'prescriptions'], []]

In [9]:
def assign_topics(dataset):
    docs = [d["text"] for d in dataset]
    topic_words = get_topics(docs)
    test_data = []
    count = 0
    for i, data in enumerate(dataset):
        txt = data["text"]
        new_txt = txt
        words = topic_words[i]
        if len(words) != 0:
            new_txt = txt + ". Topics: " + ", ".join(words)
        else:
            count += 1
        test_data.append({"text":  new_txt, "labels": data["labels"]})
    print (f"lenght of data is {len(test_data)}")
    print (f"number of input with no topics:  {count}")
    return test_data
    
test_data = assign_topics(data1)
train_data = assign_topics(data2)

Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
lenght of data is 192
number of input with no topics:  110


Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
lenght of data is 645
number of input with no topics:  328


In [10]:
with open("data/stratified_train_data_topic.json", "w") as f:
    json.dump(train_data, f, indent=4)

with open("data/stratified_test_data_topic.json", "w") as f:
    json.dump(test_data, f, indent=4)

In [6]:
# # add author feature
# # load the raw data and get the message to author dict
# file_path1 = "data/sample_new_1_14.jsonl"  # Update with your actual file path
# with open(file_path1, "r") as f:
#     data_raw1 = [json.loads(line) for line in f]
# # file_path2 = "data/message_trainset_0617_2.jsonl"  # Update with your actual file path
# # with open(file_path2, "r") as f:
# #     data_raw2 = [json.loads(line) for line in f]
# raw_data = data_raw1
# def my_clean(txt):
#     txt = txt.strip()
#     txt = " ".join(txt.split())
#     return txt
# transform = {}
# for item in raw_data:
#     from_whom = "Provider" if item["meta"]["TO_PAT_YN"] == "Y" else "Patient"
#     #sentence = my_clean(item["context"])
#     messageID = str(item["meta"]["MESSAGE_ID"])
#     author = f"From {from_whom}"
#     transform[messageID] = author

In [7]:
# len(transform)

444

In [8]:
# def check_pinfo(data):
#     count = 0
#     for item in data:
#         mid = item["messageID"]
#         if mid not in transform:
#             count += 1
#     return count
# check_pinfo(data1)

0

In [9]:
# # add author feature
# def add_pinfo(data):
#     ret = []
#     for item in data:
#         mid = item["messageID"]
#         txt = my_clean(item["text"])
#         author = transform[mid]
#         new_txt = f"{author}: {txt}"
#         ret.append({"text": new_txt, "labels": item["labels"]})
#     return ret
# new_data1 = add_pinfo(data1)
# new_data2 = add_pinfo(data2)     

In [10]:
# new_data1[1:3]

[{'text': 'From Provider: Salon pas lidocine patch to the affected area, it may take a few days to work, but use as directed',
  'labels': ['PartnershipProvider_Clinical Care',
   'SharedDecisionProvider_MakeDecision']},
 {'text': 'From Patient: Good morning,just have a question of the rybelsus the pharmacy gave me two different doses a 3 mg and a 14 mg. I start with the 3mg once a day first thing in the morning?',
  'labels': ['PartnershipPatient_Clinical care',
   'PartnershipPatient_activeParticipation/involvement',
   'PartnershipPatient_salutation',
   'SDOH_HealthCareAccessAndQuality']}]

In [11]:
# len(new_data1 + new_data2)

515

In [12]:
# # just test
# try_txt = [data1[2]["text"], data1[1]["text"]]
# # Example new sentence
# new_sentence = try_txt
# print (new_sentence)
# # Transform it with your trained topic model
# new_topics, new_probs = topic_model.transform(new_sentence)

# # Get the assigned topic ID
# for i in range(len(new_topics)):
#     assigned_topic = new_topics[i]
#     print(f"Assigned topic: {assigned_topic}")
    
#     # Get the top 3 words for this topic
#     if assigned_topic != -1:
#         topic_words = topic_model.get_topic(assigned_topic)
#         top_3_words = [word for word, _ in topic_words[:3]]
#         print(f"Top 3 words for topic {assigned_topic}: {top_3_words}")
#     else:
#         print("This sentence was classified as outlier topic (-1). No top words.")


['Good morning,just have a question of the rybelsus the pharmacy gave me two different doses a 3 mg and a 14 mg. I start with the 3mg once a day first thing in the morning?', 'Salon pas lidocine patch to the affected area, it may take a few days to work, but use as directed']


Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
Assigned topic: 7
Top 3 words for topic 7: ['mg', 'dosage', 'dose']
Assigned topic: 296
Top 3 words for topic 296: ['lidocaine', 'patches', 'lidoderm']


In [13]:
# type (new_sentence)

list

In [14]:
# # still a try
# def get_topics(sentences):
#     assert type(sentences) == list
#     # Transform it with your trained topic model
#     new_topics, new_probs = topic_model.transform(sentences)
#     # Get the assigned topic ID
#     outlier_count = 0
#     ret = []
#     for i in range(len(new_topics)):
#         assigned_topic = new_topics[i]
#         #print(f"Assigned topic: {assigned_topic}")
#         # Get the top 3 words for this topic
#         if assigned_topic != -1:
#             topic_words = topic_model.get_topic(assigned_topic)
#             top_3_words = [word for word, _ in topic_words[:3]]
#             ret.append(top_3_words)
#             #print(f"Top 3 words for topic {assigned_topic}: {top_3_words}")
#         else:
#             outlier_count += 1
#             ret.append([])
#     #print (f"number of outliers is {outlier_count}")
#     return ret
# sents = [d["text"] for d in data2]
# rets = get_topics(sents)

Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs


In [15]:
# train_data = []
# test_data = []

def assign_topics(dataset):
    docs = [d["text"] for d in dataset]
    topic_words = get_topics(docs)
    test_data = []
    count = 0
    for i, data in enumerate(dataset):
        txt = data["text"]
        new_txt = txt
        words = topic_words[i]
        if len(words) != 0:
            new_txt = txt + ". Topics: " + ", ".join(words)
        else:
            count += 1
        test_data.append({"text":  new_txt, "labels": data["labels"]})
    print (f"lenght of data is {len(test_data)}")
    print (f"number of input with no topics:  {count}")
    return test_data
    
test_data = assign_topics(new_data1)
train_data = assign_topics(new_data2)


Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
lenght of data is 108
number of input with no topics:  59


Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
lenght of data is 407
number of input with no topics:  203


In [16]:
with open("data/stratified_train_data_topic.json", "w") as f:
    json.dump(train_data, f, indent=4)

with open("data/stratified_test_data_topic.json", "w") as f:
    json.dump(test_data, f, indent=4)